# AI_Agent_Workshop_Day2_02 — Build the Service Agent

In this notebook we compare three increasingly capable systems:

1. a prompt-only baseline,
2. a retrieval-augmented baseline,
3. a tool-using Gemini agent.

The goal is to help students see why agentified design is useful.

## Setup

This notebook uses a local curated CSV/JSON retrieval path by default.

To run Gemini cells, set your API key in the environment before launching Jupyter.

In [ ]:
from pathlib import Path
import json
import os
import re
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = Path("day2_workshop")

DATA_DIR = ROOT / "data"
ARTIFACTS_DIR = ROOT / "artifacts"
catalog = pd.read_csv(DATA_DIR / "service_catalog.csv")
catalog.head()

In [ ]:
GEMINI_AVAILABLE = False

try:
    from google import genai
    from google.genai import types
    api_key = os.getenv("GEMINI_API_KEY")
    if api_key:
        client = genai.Client(api_key=api_key)
        GEMINI_AVAILABLE = True
except Exception as exc:
    print("Gemini SDK not available:", exc)

print("Gemini available:", GEMINI_AVAILABLE)

## Step 1 — Prompt-only baseline

In [ ]:
def baseline_prompt(question: str) -> str:
    return f'''
    You are a public-service routing assistant for residents in Kitchener and Waterloo Region.

    Return a JSON object with the keys:
    service_name, jurisdiction_level, responsible_body, confidence, reasoning_summary, next_steps, sources

    Question: {question}
    '''.strip()

sample_question = "Who do I contact about garbage pickup?"
print(baseline_prompt(sample_question))

In [ ]:
if GEMINI_AVAILABLE:
    baseline_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=baseline_prompt(sample_question),
    )
    print(baseline_response.text)
else:
    print("Skipping live model call. Discuss likely failure modes instead.")

## Step 2 — Local retrieval over the curated catalog

In [ ]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())

def keyword_retrieve(query: str, df: pd.DataFrame, top_k: int = 3) -> pd.DataFrame:
    q_tokens = set(tokenize(query))
    scored_rows = []
    for _, row in df.iterrows():
        text = " ".join([
            str(row.get("service_name", "")),
            str(row.get("description", "")),
            str(row.get("keywords", "")),
            str(row.get("responsible_body", "")),
        ]).lower()
        row_tokens = tokenize(text)
        overlap = sum(token in row_tokens for token in q_tokens)
        scored_rows.append((overlap, row.to_dict()))
    ranked = sorted(scored_rows, key=lambda x: x[0], reverse=True)
    ranked = [row for score, row in ranked if score > 0][:top_k]
    return pd.DataFrame(ranked)

retrieved = keyword_retrieve(sample_question, catalog, top_k=3)
retrieved

In [ ]:
def build_grounded_prompt(question: str, retrieved_df: pd.DataFrame) -> str:
    context = retrieved_df.to_dict(orient="records")
    return f'''
    You are a public-service routing assistant.

    Use the retrieved service catalog entries below as your primary evidence.
    If the evidence is ambiguous, say so.

    Retrieved context:
    {json.dumps(context, indent=2)}

    Return a JSON object with:
    service_name, jurisdiction_level, responsible_body, confidence, reasoning_summary, next_steps, sources

    Question: {question}
    '''.strip()

grounded_prompt = build_grounded_prompt(sample_question, retrieved)
print(grounded_prompt[:1200])

In [ ]:
if GEMINI_AVAILABLE:
    grounded_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=grounded_prompt,
    )
    print(grounded_response.text)
else:
    print("Skipping live grounded call.")

## Step 3 — Convert the application into a tool-using agent

In [ ]:
def search_service_index(query: str) -> list[dict]:
    results = keyword_retrieve(query, catalog, top_k=3)
    return results.to_dict(orient="records")

def lookup_service_owner(service_name: str) -> dict:
    matches = catalog[catalog["service_name"].str.lower() == service_name.lower()]
    if len(matches) == 0:
        return {
            "service_name": service_name,
            "jurisdiction_level": "Unclear",
            "responsible_body": "Unknown",
            "reasoning_summary": "No exact service match found in the local catalog.",
            "sources": [],
        }
    row = matches.iloc[0]
    return {
        "service_name": row["service_name"],
        "jurisdiction_level": row["jurisdiction_level"],
        "responsible_body": row["responsible_body"],
        "reasoning_summary": row["description"],
        "sources": [row["source_url"]],
    }

def suggest_next_steps(jurisdiction_level: str, service_name: str) -> dict:
    matches = catalog[catalog["service_name"].str.lower() == service_name.lower()]
    if len(matches) == 0:
        return {"next_steps": ["Search the relevant official service page before contacting an office."]}
    row = matches.iloc[0]
    return {
        "next_steps": [
            row["next_steps_hint"],
            f"Verify current details on the official source: {row['source_url']}"
        ]
    }

print(search_service_index("garbage pickup"))
print(lookup_service_owner("garbage pickup"))
print(suggest_next_steps("Region", "garbage pickup"))

## Gemini tool declarations

In [ ]:
if GEMINI_AVAILABLE:
    tool_declarations = [
        types.Tool(function_declarations=[{
            "name": "search_service_index",
            "description": "Search the local service catalog for relevant public services.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Resident question or service search string"}
                },
                "required": ["query"]
            }
        }]),
        types.Tool(function_declarations=[{
            "name": "lookup_service_owner",
            "description": "Look up the responsible level of government and body for a service.",
            "parameters": {
                "type": "object",
                "properties": {
                    "service_name": {"type": "string", "description": "Canonical service name"}
                },
                "required": ["service_name"]
            }
        }]),
        types.Tool(function_declarations=[{
            "name": "suggest_next_steps",
            "description": "Suggest next steps for a service after ownership is known.",
            "parameters": {
                "type": "object",
                "properties": {
                    "jurisdiction_level": {"type": "string"},
                    "service_name": {"type": "string"}
                },
                "required": ["jurisdiction_level", "service_name"]
            }
        }]),
    ]
    print(tool_declarations[0])
else:
    print("Tool declarations shown only when Gemini SDK is available.")

## Manual tool loop

In [ ]:
TOOL_REGISTRY = {
    "search_service_index": search_service_index,
    "lookup_service_owner": lookup_service_owner,
    "suggest_next_steps": suggest_next_steps,
}

def execute_tool_call(function_call):
    fn_name = function_call.name
    fn_args = dict(function_call.args)
    print("Tool selected:", fn_name)
    print("Arguments:", fn_args)
    result = TOOL_REGISTRY[fn_name](**fn_args)
    print("Tool result:", result)
    return fn_name, result

In [ ]:
question = "Who do I contact about garbage pickup?"

if GEMINI_AVAILABLE:
    initial_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f'''
        You are a public-service routing assistant for residents in Kitchener / Waterloo Region.

        Use tools when useful. Prefer grounded, structured answers.
        User question: {question}
        ''',
        config=types.GenerateContentConfig(
            tools=tool_declarations,
        ),
    )

    parts = initial_response.candidates[0].content.parts
    parts
else:
    print("Skipping live tool-call generation.")

In [ ]:
if GEMINI_AVAILABLE:
    part = initial_response.candidates[0].content.parts[0]

    if getattr(part, "function_call", None):
        fn_name, tool_result = execute_tool_call(part.function_call)
    else:
        tool_result = None
        print("No function call returned.")
        print(initial_response.text)

## Return the tool result back to the model

In [ ]:
if GEMINI_AVAILABLE and tool_result is not None:
    final_response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[
            types.Content(
                role="user",
                parts=[types.Part(text=question)]
            ),
            types.Content(
                role="model",
                parts=[part]
            ),
            types.Content(
                role="user",
                parts=[
                    types.Part.from_function_response(
                        name=fn_name,
                        response=tool_result
                    )
                ]
            )
        ],
    )

    print(final_response.text)
else:
    print("Skipping final synthesis call.")

## Exercise

1. Add a new tool called `retrieve_service_docs`.
2. Make the final answer always include at least one official source URL.
3. Create a question that requires hedging or clarification.
4. Compare the prompt-only baseline against the tool-using version on five examples.